#### PARTIE 1 - IMPORTS ET CHARGEMENT DES DONNÉES

In [2]:
import os
import re
import random
import pandas as pd
from textblob import TextBlob
import spacy
from nltk.corpus import wordnet
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to C:\Users\fatima
[nltk_data]     ezzahra\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\fatima
[nltk_data]     ezzahra\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
# Charger les datasets
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (10003, 2)
Test shape: (3080, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
# Afficher les catégories et leur nombre
categories = train['category'].unique()
print("Nombre d'intentions :", len(categories))
print("Intents:", categories)

Nombre d'intentions : 77
Intents: ['card_arrival' 'card_linking' 'exchange_rate'
 'card_payment_wrong_exchange_rate' 'extra_charge_on_statement'
 'pending_cash_withdrawal' 'fiat_currency_support'
 'card_delivery_estimate' 'automatic_top_up' 'card_not_working'
 'exchange_via_app' 'lost_or_stolen_card' 'age_limit' 'pin_blocked'
 'contactless_not_working' 'top_up_by_bank_transfer_charge'
 'pending_top_up' 'cancel_transfer' 'top_up_limits'
 'wrong_amount_of_cash_received' 'card_payment_fee_charged'
 'transfer_not_received_by_recipient' 'supported_cards_and_currencies'
 'getting_virtual_card' 'card_acceptance' 'top_up_reverted'
 'balance_not_updated_after_cheque_or_cash_deposit'
 'card_payment_not_recognised' 'edit_personal_details'
 'why_verify_identity' 'unable_to_verify_identity' 'get_physical_card'
 'visa_or_mastercard' 'topping_up_by_card' 'disposable_card_limits'
 'compromised_card' 'atm_support' 'direct_debit_payment_not_recognised'
 'passcode_forgotten' 'declined_cash_withdrawal' 'p

In [5]:
# Distribution initiale des catégories
print("\nDistribution initiale des catégories dans TRAIN :")
train_counts = train['category'].value_counts().reset_index()
train_counts.columns = ['category', 'Count']
print(train_counts)


Distribution initiale des catégories dans TRAIN :
                                            category  Count
0                           card_payment_fee_charged    187
1                direct_debit_payment_not_recognised    182
2   balance_not_updated_after_cheque_or_cash_deposit    181
3                      wrong_amount_of_cash_received    180
4                             cash_withdrawal_charge    177
..                                               ...    ...
72                               lost_or_stolen_card     82
73                                    card_swallowed     61
74                                   card_acceptance     59
75                          virtual_card_not_working     41
76                           contactless_not_working     35

[77 rows x 2 columns]


#### PARTIE 2 - AUGMENTATION DES DONNÉES

In [6]:
# Trouver la catégorie majoritaire pour équilibrer
counts = train['category'].value_counts()
max_count = counts.max()

# Fonctions d'augmentation
def synonym_replacement(text, n=2):
    """Remplace aléatoirement n mots par leurs synonymes WordNet"""
    words = text.split()
    new_words = words.copy()
    random_idx = random.sample(range(len(words)), min(n, len(words)))
    for idx in random_idx:
        synonyms = wordnet.synsets(words[idx])
        if synonyms:
            lemmas = synonyms[0].lemma_names()
            if lemmas:
                new_words[idx] = lemmas[0].replace("_", " ")
    return " ".join(new_words)

def random_swap(text, n=1):
    """Échange aléatoirement l’ordre de mots"""
    words = text.split()
    for _ in range(n):
        if len(words) > 2:
            i, j = random.sample(range(len(words)), 2)
            words[i], words[j] = words[j], words[i]
    return " ".join(words)

def augment_text(text):
    """Choisit une technique d'augmentation aléatoire"""
    choice = random.choice(['synonym', 'swap', 'none'])
    if choice == 'synonym':
        return synonym_replacement(text)
    elif choice == 'swap':
        return random_swap(text)
    else:
        return text

# Générer des données augmentées pour les catégories minoritaires
augmented_rows = []
for cat, count in counts.items():
    if count < max_count:
        needed = max_count - count
        subset = train[train['category'] == cat]
        print(f"Augmentation catégorie '{cat}' ({count} → {max_count})")
        for _ in range(needed):
            original = subset.sample(1).iloc[0]
            new_text = augment_text(original['text'])
            augmented_rows.append({'text': new_text, 'category': cat})



Augmentation catégorie 'direct_debit_payment_not_recognised' (182 → 187)
Augmentation catégorie 'balance_not_updated_after_cheque_or_cash_deposit' (181 → 187)
Augmentation catégorie 'wrong_amount_of_cash_received' (180 → 187)
Augmentation catégorie 'cash_withdrawal_charge' (177 → 187)
Augmentation catégorie 'transaction_charged_twice' (175 → 187)
Augmentation catégorie 'declined_cash_withdrawal' (173 → 187)
Augmentation catégorie 'transfer_fee_charged' (172 → 187)
Augmentation catégorie 'balance_not_updated_after_bank_transfer' (171 → 187)
Augmentation catégorie 'transfer_not_received_by_recipient' (171 → 187)
Augmentation catégorie 'request_refund' (169 → 187)
Augmentation catégorie 'card_payment_not_recognised' (168 → 187)
Augmentation catégorie 'card_payment_wrong_exchange_rate' (167 → 187)
Augmentation catégorie 'extra_charge_on_statement' (166 → 187)
Augmentation catégorie 'wrong_exchange_rate_for_cash_withdrawal' (163 → 187)
Augmentation catégorie 'Refund_not_showing_up' (162 → 1

In [7]:
# Fusionner et sauvegarder
augmented_df = pd.DataFrame(augmented_rows)
train_aug = pd.concat([train, augmented_df], ignore_index=True)
os.makedirs("../data/augmented", exist_ok=True)
train_aug.to_csv("../data/augmented/train_augmented.csv", index=False)

print("Nouvelle distribution après augmentation :")
print(train_aug['category'].value_counts())

Nouvelle distribution après augmentation :
category
card_arrival                        187
card_linking                        187
exchange_rate                       187
card_payment_wrong_exchange_rate    187
extra_charge_on_statement           187
                                   ... 
cash_withdrawal_charge              187
card_about_to_expire                187
apple_pay_or_google_pay             187
verify_my_identity                  187
country_support                     187
Name: count, Length: 77, dtype: int64


In [8]:
# Charger dataset augmenté
train = pd.read_csv("../data/augmented/train_augmented.csv")
print("Train shape après augmentation:", train.shape)
train.head(-10)

Train shape après augmentation: (14399, 2)


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival
...,...,...
14384,Why wasn't my contactless accepted at the metro?,contactless_not_working
14385,anywhere need help fixing my contactless. It's...,contactless_not_working
14386,Are my contactless settings correct? I tried t...,contactless_not_working
14387,Would reinstalling the app solve the problem?,contactless_not_working


#### PARTIE 3 - NETTOYAGE DU TEXTE


In [10]:
# Calcul métriques de qualité
def text_metrics(df, name="Dataset"):
    nan_count = df['text'].isnull().sum()
    non_text = df['text'].apply(lambda x: not isinstance(x, str)).sum()
    too_short = df['text'].apply(lambda x: isinstance(x, str) and len(x.strip()) <= 2).sum()
    total = len(df)
    print(f"{name} :")
    print(f" - Lignes avec texte manquant: {nan_count}")
    print(f" - Lignes non-texte: {non_text}")
    print(f" - Lignes texte trop court: {too_short}")
    print(f" - Total lignes: {total}\n")

text_metrics(train, "TRAIN")
text_metrics(test, "TEST")

TRAIN :
 - Lignes avec texte manquant: 0
 - Lignes non-texte: 0
 - Lignes texte trop court: 0
 - Total lignes: 14399

TEST :
 - Lignes avec texte manquant: 0
 - Lignes non-texte: 0
 - Lignes texte trop court: 0
 - Total lignes: 3080



In [9]:
# Fonction de nettoyage
def clean_text(text):
    """
    Nettoyage simple :
    - minuscule
    - suppression ponctuation
    - suppression espaces multiples
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\d+", "<NUM>", text)

    return text

In [10]:
# Appliquer nettoyage
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

In [11]:
print("Exemple après nettoyage :")
train[['text','clean_text']].head()

Exemple après nettoyage :


,text,clean_text
0,I am still waiting on my card?,i am still waiting on my card
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasnt arrived a...
2,I have been waiting over a week. Is the card s...,i have been waiting over a week is the card st...
3,Can I track my card while it is in the process...,can i track my card while it is in the process...
4,"How do I know if I will get my card, or if it ...",how do i know if i will get my card or if it i...


In [12]:
# Normalisation des contractions
contractions = {
    "can't": "cannot",
    "won't": "will not",
    "i'm": "i am",
    "it's": "it is",
    "doesn't": "does not",
    "didn't": "did not",
    "hasn't": "has not",
    "haven't": "have not",
    "i've": "i have",
    "you're": "you are"
}

def normalize_contractions(text):
    for c, full in contractions.items():
        text = text.replace(c, full)
    return text

train['clean_text'] = train['clean_text'].apply(normalize_contractions)
test['clean_text'] = test['clean_text'].apply(normalize_contractions)

print("Exemple après normalisation des contractions :")
train[['text','clean_text']].head()

Exemple après normalisation des contractions :


,text,clean_text
0,I am still waiting on my card?,i am still waiting on my card
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasnt arrived a...
2,I have been waiting over a week. Is the card s...,i have been waiting over a week is the card st...
3,Can I track my card while it is in the process...,can i track my card while it is in the process...
4,"How do I know if I will get my card, or if it ...",how do i know if i will get my card or if it i...


#### PARTIE 4 - LEMMATISATION ET ANALYSE SYNTAXIQUE


In [13]:
nlp = spacy.load("en_core_web_sm")

def analyze_syntax(text):
    """
    Applique :
    - Tokenisation
    - Suppression des stopwords
    - Lemmatisation
    - Filtrage des caractères spéciaux
    Retourne le texte lemmatisé nettoyé.
    """
    if not isinstance(text, str):
        return ""
    
    doc = nlp(text)
    
    lemmas = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    
    return " ".join(lemmas)


In [14]:
# Appliquer sur train et test
print("Application du prétraitement syntaxique (lemmatisation)...")
train['clean_text'] = train['clean_text'].apply(analyze_syntax)
test['clean_text'] = test['clean_text'].apply(analyze_syntax)

Application du prétraitement syntaxique (lemmatisation)...


In [20]:
print("Exemple après lemmatisation :")
train[['text','clean_text']].head()

Exemple après lemmatisation :


,text,clean_text
0,I am still waiting on my card?,wait card
1,What can I do if my card still hasn't arrived ...,card not arrive NUM week
2,I have been waiting over a week. Is the card s...,wait week card come
3,Can I track my card while it is in the process...,track card process delivery
4,"How do I know if I will get my card, or if it ...",know card lose


In [21]:
# Sauvegarder les datasets prétraités
train.to_csv("../data/processed/train_preprocessed.csv", index=False)
test.to_csv("../data/processed/test_preprocessed.csv", index=False)

print("Données prétraitées sauvegardées dans '../data/processed/'")


Données prétraitées sauvegardées dans '../data/processed/'
